# 12 ? Mini Project: Prediction Pipeline

## Project brief
Build and evaluate a model to predict house prices from synthetic features.


In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score

rng = np.random.default_rng(123)
n = 500

df = pd.DataFrame({
    "sqft": rng.normal(1600, 500, n).clip(400, 5000),
    "bedrooms": rng.integers(1, 6, n),
    "age": rng.integers(0, 60, n),
    "neighborhood": rng.choice(["A", "B", "C", "D"], n),
})

base = 50000 + df["sqft"] * 180 + df["bedrooms"] * 10000 - df["age"] * 800
neigh_boost = df["neighborhood"].map({"A": 90000, "B": 50000, "C": 20000, "D": 0})
noise = rng.normal(0, 25000, n)

df["price"] = base + neigh_boost + noise

df.head()

In [ ]:
X = df.drop(columns=["price"])
y = df["price"]

num_cols = ["sqft", "bedrooms", "age"]
cat_cols = ["neighborhood"]

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore")),
])

preprocess = ColumnTransformer([
    ("num", numeric_pipeline, num_cols),
    ("cat", categorical_pipeline, cat_cols),
])

model = RandomForestRegressor(n_estimators=200, random_state=42)

pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", model),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pipe.fit(X_train, y_train)
pred = pipe.predict(X_test)

print("MAE:", round(mean_absolute_error(y_test, pred), 2))
print("R2:", round(r2_score(y_test, pred), 3))

In [ ]:
# Try GradientBoostingRegressor
gb_model = GradientBoostingRegressor(random_state=42)
gb_pipe = Pipeline([
    ("preprocess", preprocess),
    ("model", gb_model),
])

gb_pipe.fit(X_train, y_train)
gb_pred = gb_pipe.predict(X_test)
print("GB MAE:", round(mean_absolute_error(y_test, gb_pred), 2))
print("GB R2:", round(r2_score(y_test, gb_pred), 3))

# Add cross-validation
cv = KFold(n_splits=5, shuffle=True, random_state=42)
rf_cv_mae = -cross_val_score(pipe, X, y, cv=cv, scoring="neg_mean_absolute_error")
rf_cv_r2 = cross_val_score(pipe, X, y, cv=cv, scoring="r2")
gb_cv_mae = -cross_val_score(gb_pipe, X, y, cv=cv, scoring="neg_mean_absolute_error")
gb_cv_r2 = cross_val_score(gb_pipe, X, y, cv=cv, scoring="r2")

print("\nCV (5-fold)")
print("RF MAE mean/std:", round(rf_cv_mae.mean(), 2), "/", round(rf_cv_mae.std(), 2))
print("RF R2  mean/std:", round(rf_cv_r2.mean(), 3), "/", round(rf_cv_r2.std(), 3))
print("GB MAE mean/std:", round(gb_cv_mae.mean(), 2), "/", round(gb_cv_mae.std(), 2))
print("GB R2  mean/std:", round(gb_cv_r2.mean(), 3), "/", round(gb_cv_r2.std(), 3))

# Inspect Random Forest feature importances
feature_names = pipe.named_steps["preprocess"].get_feature_names_out()
importances = pipe.named_steps["model"].feature_importances_
fi = pd.DataFrame({"feature": feature_names, "importance": importances}).sort_values("importance", ascending=False)
print("\nTop feature importances (RF):")
print(fi.head(10).reset_index(drop=True))
